In [1]:
import os
import xml.etree.ElementTree as ET

In [2]:
IMAGE_DIR = "car_plate/images"
ANNOTATION_DIR = "car_plate/annotations"
OUTPUT_LABEL_DIR = "Output1/labels"
os.makedirs(OUTPUT_LABEL_DIR, exist_ok=True)

In [3]:
for xml_file in os.listdir(ANNOTATION_DIR):
    xml_path = os.path.join(ANNOTATION_DIR, xml_file)
    if not os.path.isfile(xml_path):
        continue
    if not xml_file.lower().endswith(".xml"):
        continue
    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()
        filename = root.find("filename").text
        size = root.find("size")
        w = int(size.find("width").text)
        h = int(size.find("height").text)
        txt_filename = os.path.splitext(filename)[0] + ".txt"
        txt_path = os.path.join(OUTPUT_LABEL_DIR, txt_filename)
        with open(txt_path, "w") as f:
            for obj in root.findall("object"):
                bbox = obj.find("bndbox")
                xmin = int(bbox.find("xmin").text)
                ymin = int(bbox.find("ymin").text)
                xmax = int(bbox.find("xmax").text)
                ymax = int(bbox.find("ymax").text)
                x_center = ((xmin + xmax) / 2) / w
                y_center = ((ymin + ymax) / 2) / h
                width = (xmax - xmin) / w
                height = (ymax - ymin) / h
                f.write(
                    f"0 {x_center} {y_center} {width} {height}\n"
                )
    except ET.ParseError:
        print(f"Skipping invalid XML: {xml_file}")

In [4]:
import os
import shutil
import random

In [5]:
IMAGE_SRC = "car_plate/images"
LABEL_SRC = "Output1/labels"
BASE_DIR = "Output1"
for folder in ["images/train", "images/val", "labels/train", "labels/val"]:
    os.makedirs(os.path.join(BASE_DIR, folder), exist_ok=True)
images = os.listdir(IMAGE_SRC)
random.shuffle(images)
split = int(0.8 * len(images))
train_imgs = images[:split]
val_imgs = images[split:]

In [6]:
for folder in [
    "images/train",
    "images/val",
    "labels/train",
    "labels/val"
]:
    os.makedirs(os.path.join(BASE_DIR, folder), exist_ok=True)
valid_extensions = (".jpg", ".jpeg", ".png", ".bmp")
images = [
    img for img in os.listdir(IMAGE_SRC)
    if os.path.isfile(os.path.join(IMAGE_SRC, img))
    and img.lower().endswith(valid_extensions)
]
print(f"Total valid images found: {len(images)}")

Total valid images found: 433


In [7]:
images_with_labels = []
for img in images:
    image_name = os.path.splitext(img)[0]
    label = image_name + ".txt"
    label_path = os.path.join(LABEL_SRC, label)
    if os.path.isfile(label_path):
        images_with_labels.append(img)
    else:
        print(f"Warning: No label found for {img}")
print(f"Images with corresponding labels: {len(images_with_labels)}")

Images with corresponding labels: 433


In [8]:
random.shuffle(images_with_labels)
split = int(0.8 * len(images_with_labels))
train_imgs = images_with_labels[:split]
val_imgs = images_with_labels[split:]
print(f"Training images   : {len(train_imgs)}")
print(f"Validation images : {len(val_imgs)}")

Training images   : 346
Validation images : 87


In [9]:
def copy_files(img_list, split_type):
    for img in img_list:
        image_src_path = os.path.join(
            IMAGE_SRC,
            img
        )
        image_dst_path = os.path.join(
            BASE_DIR,
            f"images/{split_type}",
            img
        )
        image_name = os.path.splitext(img)[0]
        label = image_name + ".txt"
        label_src_path = os.path.join(
            LABEL_SRC,
            label
        )
        label_dst_path = os.path.join(
            BASE_DIR,
            f"labels/{split_type}",
            label
        )
        shutil.copy2(
            image_src_path,
            image_dst_path
        )
        shutil.copy2(
            label_src_path,
            label_dst_path
        )
copy_files(train_imgs, "train")
copy_files(val_imgs, "val")

In [10]:
from ultralytics import YOLO
model = YOLO("yolov8n.pt")
model.train(
    data="plate.yaml",
    epochs=10,
    imgsz=640
)

New https://pypi.org/project/ultralytics/8.4.153 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.129  Python-3.12.8 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4070 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=plate.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=tra

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x00000190B82E4920>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.0480